# Warfarin QML — Main benchmark k=4 | QSVC C-grid

**Mục tiêu:** Khảo sát độ nhạy theo `C ∈ {0.1, 1.0, 10.0}` của bộ phân loại Quantum Kernel, dùng **cùng 600 mẫu / cùng 5 folds / cùng PCA k=4** với benchmark đã thực hiện. **Không dùng Final Test**.

**Lưu ý nghiên cứu:** `docs/experiment_matrix.csv` hiện chưa khóa danh sách C cụ thể; `C=1` đã chạy trước khi bổ sung grid. Vì vậy C-grid này có trạng thái `EXPLORATORY_POST_BASELINE`, không phải tuning được preregister. Sử dụng Macro-F1 trên development CV để so sánh, **không được tuyên bố điểm số của C được chọn là đánh giá không thiên lệch sau tuning**, cũng chưa khóa final config.

**Kernel reuse:** Tạo một `FidelityStatevectorKernel` mới cho mỗi fold, tính `K_train` (480×480) và `K_val` (120×480) **một lần cho toàn bộ grid**, rồi dùng `sklearn.svm.SVC(kernel="precomputed")`. Đây là triển khai tương đương cách QSVC gọi SVC với kernel lượng tử; Cell kiểm tra C=1 dựa trên prediction đã lưu từ QSVC gốc.

**Lưu kết quả:** `docs/results/qsvc_k4_cgrid_{metrics,predictions,confusion,summary}.csv` và `qsvc_k4_cgrid_config.json`. Không ghi đè bất kỳ kết quả nào trước đó, không tạo thêm file trong `scripts/`.

**Chạy:** VS Code → notebook kernel Conda **`nt409`** → `Run All` sau khi đọc phần kiểm tra. Toàn bộ số liệu được tính mới trong notebook trên máy bạn; notebook cung cấp chưa được chạy trên dữ liệu của bạn.


## Cell 1 — Imports, tìm project root và khóa output

Cấu hình C-grid chỉ định ở đây. Script dừng nếu các file output đã tồn tại để tránh làm mất kết quả cũ.

In [1]:
from pathlib import Path
from time import perf_counter
from importlib.metadata import version
import sys
import hashlib
import json

import numpy as np
import pandas as pd
import yaml
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, recall_score, confusion_matrix
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityStatevectorKernel


def find_project_root(start):
    for path in (start.resolve(), *start.resolve().parents):
        if ((path / "configs/feature_policy.yaml").is_file()
                and (path / "data/splits/quantum_dev_folds.csv").is_file()):
            return path
    raise FileNotFoundError("Open this notebook from inside Warfarin project")

ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.prepare_stage4 import load_and_validate_dataset, verify_raw_file
from src.preprocessing import build_preprocessor

OUT = ROOT / "docs/results"
SPLIT_DIR = ROOT / "data/splits"
CLASSES = ["Low", "Intermediate", "High"]
C_GRID = (0.1, 1.0, 10.0)
assert len(C_GRID) == len(set(C_GRID)) == 3 and 1.0 in C_GRID

PREFIX = "qsvc_k4_cgrid"
FILES = {
    "metrics": OUT / f"{PREFIX}_metrics.csv",
    "predictions": OUT / f"{PREFIX}_predictions.csv",
    "confusion": OUT / f"{PREFIX}_confusion.csv",
    "summary": OUT / f"{PREFIX}_summary.csv",
    "config": OUT / f"{PREFIX}_config.json",
}
existing = [str(p) for p in FILES.values() if p.exists()]
if existing:
    raise FileExistsError(f"Existing benchmark outputs; not overwriting: {existing}")

BASELINE = OUT / "qsvc_k4_5fold_predictions.csv"
assert BASELINE.is_file(), f"Need original QSVC C=1 predictions for equivalence check: {BASELINE}"
print("[PASS] Environment:", sys.executable)
print("[PASS] Project:", ROOT)
print("[PASS] C-grid:", C_GRID)
print("[PASS] No output collisions")


[PASS] Environment: c:\Users\admin\anaconda3\envs\nt409\python.exe
[PASS] Project: D:\Warfarin-Quantum-Machine-Learning-Classification
[PASS] C-grid: (0.1, 1.0, 10.0)
[PASS] No output collisions


## Cell 2 — Đọc và xác minh dữ liệu đã khóa

Đảm bảo 600 Quantum Development samples chỉ thuộc Development, không nằm trong Final Test; kiểm tra hashes và class labels.

In [2]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

verify_raw_file()
with (ROOT / "configs/feature_policy.yaml").open(encoding="utf-8") as f:
    policy = yaml.safe_load(f)
assert policy["training_allowed"] is True
approved = list(policy["approved_X_columns"])
assert len(approved) == len(set(approved)) == 10

manifest_path = SPLIT_DIR / "split_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest["status"] == "LOCKED_BEFORE_MODEL_PERFORMANCE"
assert manifest["approved_features"] == approved
for filename, expected in manifest["files"].items():
    assert sha256_file(SPLIT_DIR / filename) == expected, f"Hash mismatch: {filename}"

X, y = load_and_validate_dataset(approved)
labels = pd.read_csv(ROOT / "data/interim/labels.csv")
folds = pd.read_csv(SPLIT_DIR / "quantum_dev_folds.csv")
splits = pd.read_csv(SPLIT_DIR / "final_split.csv")
y_array = np.asarray(y, dtype=str)

assert len(X) == len(y_array) == len(labels) == 5410
assert labels["raw_row_index"].is_unique
assert folds["raw_row_index"].is_unique
assert np.array_equal(y_array, labels["target_group"].astype(str).to_numpy())
assert set(folds["fold_id"].astype(int)) == set(range(5))
qdev_ids = set(folds["raw_row_index"].astype(int))
dev_ids = set(splits.loc[splits["partition"] == "development", "raw_row_index"].astype(int))
test_ids = set(splits.loc[splits["partition"] == "final_test", "raw_row_index"].astype(int))
assert len(qdev_ids) == 600
assert qdev_ids.issubset(dev_ids) and qdev_ids.isdisjoint(test_ids)
assert dev_ids.isdisjoint(test_ids)
assert dev_ids | test_ids == set(labels["raw_row_index"].astype(int))

BASELINE_DF = pd.read_csv(BASELINE)
assert set(["raw_row_index", "fold", "actual", "predicted"]).issubset(BASELINE_DF.columns)
assert len(BASELINE_DF) == 600 and BASELINE_DF["raw_row_index"].is_unique
print("[PASS] Raw/source & 4 split hashes")
print("[PASS] 600 qdev samples and 5 locked folds")
print("[PASS] Final Test isolated")
print("[PASS] Original QSVC C=1 baseline predictions available")


[PASS] Raw/source & 4 split hashes
[PASS] 600 qdev samples and 5 locked folds
[PASS] Final Test isolated
[PASS] Original QSVC C=1 baseline predictions available


## Cell 3 — Hàm tạo kernel matrix theo training fold

Mọi phép biến đổi học tham số trên training fold duy nhất. `FidelityStatevectorKernel` dùng `shots=None` (exact statevector simulator), `auto_clear_cache=False` để tái sử dụng statevectors giữa lần tính train và validation. **Không lưu kernel matrix lớn vào disk**; chỉ giữ tạm trong RAM khi đang chạy một fold.


In [3]:
def prepare_fold_kernel(fold_id):
    val_ids = set(folds.loc[folds["fold_id"] == fold_id, "raw_row_index"].astype(int))
    train_ids = qdev_ids - val_ids
    assert len(train_ids) == 480 and len(val_ids) == 120
    assert (train_ids | val_ids) == qdev_ids and train_ids.isdisjoint(val_ids)
    assert (train_ids | val_ids).isdisjoint(test_ids)

    train_pos = np.flatnonzero(labels["raw_row_index"].isin(train_ids).to_numpy())
    val_pos = np.flatnonzero(labels["raw_row_index"].isin(val_ids).to_numpy())
    X_train, X_val = X.iloc[train_pos], X.iloc[val_pos]
    y_train, y_val = y_array[train_pos], y_array[val_pos]

    preprocessor = build_preprocessor(approved)
    encoded_train = preprocessor.fit_transform(X_train)
    encoded_val = preprocessor.transform(X_val)
    pca = PCA(n_components=4, svd_solver="full")
    pca_train = pca.fit_transform(encoded_train)
    pca_val = pca.transform(encoded_val)
    scaler = MinMaxScaler(feature_range=(0.0, float(np.pi)))
    z_train = np.clip(scaler.fit_transform(pca_train), 0.0, np.pi)
    z_val = np.clip(scaler.transform(pca_val), 0.0, np.pi)
    assert z_train.shape == (480, 4) and z_val.shape == (120, 4)
    assert np.isfinite(z_train).all() and np.isfinite(z_val).all()

    fm = zz_feature_map(feature_dimension=4, reps=2, entanglement="linear")
    kernel = FidelityStatevectorKernel(
        feature_map=fm, shots=None, auto_clear_cache=False
    )

    tic = perf_counter()
    K_train = kernel.evaluate(x_vec=z_train)
    kernel_train_seconds = perf_counter() - tic
    tic = perf_counter()
    K_val = kernel.evaluate(x_vec=z_val, y_vec=z_train)
    kernel_val_seconds = perf_counter() - tic

    assert K_train.shape == (480, 480) and K_val.shape == (120, 480)
    assert np.isfinite(K_train).all() and np.isfinite(K_val).all()
    assert np.allclose(K_train, K_train.T, atol=1e-9)
    assert np.allclose(np.diag(K_train), 1.0, atol=1e-9)
    assert K_train.min() >= -1e-7 and K_val.min() >= -1e-7
    assert K_train.max() <= 1 + 1e-7 and K_val.max() <= 1 + 1e-7

    return {
        "fold": fold_id,
        "K_train": K_train,
        "K_val": K_val,
        "y_train": y_train,
        "y_val": y_val,
        "val_pos": val_pos,
        "kernel_train_seconds": kernel_train_seconds,
        "kernel_val_seconds": kernel_val_seconds,
        "pca_explained_variance": float(pca.explained_variance_ratio_.sum()),
    }


## Cell 4 — Preflight tương đương `C=1` trên Fold 0

Cực kỳ quan trọng trước khi grid: so sánh từng prediction trên Fold 0 giữa `SVC(precomputed)` và QSVC gốc đã lưu. **Nếu không khớp 120/120, notebook dừng, không tạo kết quả mới.**

In [4]:
fold0_data = prepare_fold_kernel(0)
preflight_model = SVC(kernel="precomputed", C=1.0, class_weight=None)
preflight_model.fit(fold0_data["K_train"], fold0_data["y_train"])
preflight_pred = preflight_model.predict(fold0_data["K_val"])
preflight_df = pd.DataFrame({
    "raw_row_index": labels.iloc[fold0_data["val_pos"]]["raw_row_index"].astype(int).to_numpy(),
    "actual": fold0_data["y_val"],
    "predicted_precomputed": preflight_pred,
})
ref0 = BASELINE_DF.loc[BASELINE_DF["fold"] == 0, ["raw_row_index", "actual", "predicted"]]
matched = preflight_df.merge(ref0, on="raw_row_index", validate="one_to_one")
assert len(matched) == 120
assert (matched["actual_x"] == matched["actual_y"]).all()
match_count = int((matched["predicted_precomputed"] == matched["predicted"]).sum())
print(f"C=1 precomputed vs old QSVC: {match_count}/120 identical predictions")
if match_count != 120:
    bad = matched.loc[matched["predicted_precomputed"] != matched["predicted"]]
    display(bad.head(10))
    raise RuntimeError("C=1 kernel implementation differs from baseline: STOP before full grid")
print("[PASS] C=1 Fold 0 exact prediction agreement")
print("[PASS] Kernel shapes: 480×480 train, 120×480 validation")
print("Kernel construction seconds:", round(fold0_data["kernel_train_seconds"], 3),
      round(fold0_data["kernel_val_seconds"], 3))


C=1 precomputed vs old QSVC: 120/120 identical predictions
[PASS] C=1 Fold 0 exact prediction agreement
[PASS] Kernel shapes: 480×480 train, 120×480 validation
Kernel construction seconds: 1.24 0.291


## Cell 5 — Full C-grid × 5 folds

Chạy **15 lần SVC fit**, nhưng chỉ tính kernel **5 lần** cho train và **5 lần** cho validation. Fold 0 tái sử dụng kernel vừa tính ở preflight. *Không dùng class weighting trong core*. Toàn bộ kết quả nằm trong RAM tới khi Cell 6 xác minh và ghi file.

In [5]:
metrics_rows, prediction_rows, confusion_rows = [], [], []
for fold_id in range(5):
    data = fold0_data if fold_id == 0 else prepare_fold_kernel(fold_id)
    y_train, y_val = data["y_train"], data["y_val"]
    K_train, K_val = data["K_train"], data["K_val"]
    ids_val = labels.iloc[data["val_pos"]]["raw_row_index"].astype(int).to_numpy()

    for C in C_GRID:
        svc = SVC(kernel="precomputed", C=float(C), class_weight=None)
        tic = perf_counter()
        svc.fit(K_train, y_train)
        svc_fit_seconds = perf_counter() - tic
        tic = perf_counter()
        predicted = np.asarray(svc.predict(K_val), dtype=str)
        svc_predict_seconds = perf_counter() - tic

        rec = recall_score(y_val, predicted, labels=CLASSES, average=None, zero_division=0)
        cm = confusion_matrix(y_val, predicted, labels=CLASSES)
        metrics_rows.append({
            "fold": int(fold_id), "C": float(C), "k": 4,
            "train_n": 480, "val_n": 120,
            "macro_f1": float(f1_score(y_val, predicted, labels=CLASSES, average="macro", zero_division=0)),
            "accuracy": float(accuracy_score(y_val, predicted)),
            "balanced_accuracy": float(balanced_accuracy_score(y_val, predicted)),
            "recall_low": float(rec[0]), "recall_mid": float(rec[1]), "recall_high": float(rec[2]),
            "pred_high": int(np.sum(predicted == "High")),
            "support_vectors": int(svc.n_support_.sum()),
            "pca_explained_variance": data["pca_explained_variance"],
            "kernel_train_seconds": data["kernel_train_seconds"],
            "kernel_val_seconds": data["kernel_val_seconds"],
            "svc_fit_seconds": svc_fit_seconds, "svc_predict_seconds": svc_predict_seconds,
        })
        for rid, actual, guess in zip(ids_val, y_val, predicted):
            prediction_rows.append({
                "fold": int(fold_id), "C": float(C), "raw_row_index": int(rid),
                "actual": str(actual), "predicted": str(guess)
            })
        for i, actual in enumerate(CLASSES):
            for j, guess in enumerate(CLASSES):
                confusion_rows.append({
                    "scope": "fold", "fold": int(fold_id), "C": float(C),
                    "actual": actual, "predicted": guess, "count": int(cm[i, j]),
                })
        print(f"[PASS] Fold={fold_id}, C={C:g}: F1={metrics_rows[-1]['macro_f1']:.4f}, "
              f"Acc={metrics_rows[-1]['accuracy']:.4f}, RecallHigh={metrics_rows[-1]['recall_high']:.4f}",
              flush=True)
    del K_train, K_val
    if fold_id != 0:
        del data

metrics_df = pd.DataFrame(metrics_rows)
predictions_df = pd.DataFrame(prediction_rows)
print("[PASS] 15 model-fold fits complete; 1,800 validation predictions")


[PASS] Fold=0, C=0.1: F1=0.2342, Acc=0.5417, RecallHigh=0.0000
[PASS] Fold=0, C=1: F1=0.3762, Acc=0.5833, RecallHigh=0.0000
[PASS] Fold=0, C=10: F1=0.4495, Acc=0.5250, RecallHigh=0.2667
[PASS] Fold=1, C=0.1: F1=0.2342, Acc=0.5417, RecallHigh=0.0000
[PASS] Fold=1, C=1: F1=0.3629, Acc=0.5750, RecallHigh=0.0000
[PASS] Fold=1, C=10: F1=0.4167, Acc=0.5250, RecallHigh=0.2000
[PASS] Fold=2, C=0.1: F1=0.2342, Acc=0.5417, RecallHigh=0.0000
[PASS] Fold=2, C=1: F1=0.3685, Acc=0.5833, RecallHigh=0.0000
[PASS] Fold=2, C=10: F1=0.4815, Acc=0.5417, RecallHigh=0.3333
[PASS] Fold=3, C=0.1: F1=0.2342, Acc=0.5417, RecallHigh=0.0000
[PASS] Fold=3, C=1: F1=0.3323, Acc=0.5083, RecallHigh=0.0000
[PASS] Fold=3, C=10: F1=0.4055, Acc=0.5000, RecallHigh=0.1875
[PASS] Fold=4, C=0.1: F1=0.2366, Acc=0.5500, RecallHigh=0.0000
[PASS] Fold=4, C=1: F1=0.3614, Acc=0.5667, RecallHigh=0.0000
[PASS] Fold=4, C=10: F1=0.3588, Acc=0.4500, RecallHigh=0.1333
[PASS] 15 model-fold fits complete; 1,800 validation predictions


## Cell 6 — QC toàn bộ, tổng hợp và lưu kết quả

So sánh dự đoán `C=1` với **toàn bộ** 600 predictions QSVC đã có. Nếu lệch, dừng để phân tích; không ghi đè file cũ. `suggested_C_by_CV` chỉ là **ứng viên** theo Macro-F1, chưa chốt cấu hình cuối.

In [6]:
assert len(metrics_df) == 15
assert len(predictions_df) == 1800
assert set(metrics_df["fold"]) == set(range(5))
assert set(metrics_df["C"]) == set(C_GRID)
assert not metrics_df.duplicated(["fold", "C"]).any()
assert not predictions_df.duplicated(["C", "raw_row_index"]).any()

for C in C_GRID:
    this = predictions_df.loc[predictions_df["C"] == C]
    assert len(this) == 600 and set(this["raw_row_index"]) == qdev_ids
    assert not set(this["raw_row_index"]).intersection(test_ids)
    for fold_id in range(5):
        subset = this.loc[this["fold"] == fold_id]
        expected = set(folds.loc[folds["fold_id"] == fold_id, "raw_row_index"].astype(int))
        assert len(subset) == 120 and set(subset["raw_row_index"]) == expected

c1 = predictions_df.loc[predictions_df["C"] == 1.0, ["raw_row_index", "fold", "actual", "predicted"]]
compare = c1.merge(BASELINE_DF, on=["raw_row_index", "fold"], validate="one_to_one", suffixes=("_grid", "_qsvc"))
assert len(compare) == 600
assert (compare["actual_grid"] == compare["actual_qsvc"]).all()
agreement = int((compare["predicted_grid"] == compare["predicted_qsvc"]).sum())
print("=== C=1 EQUIVALENCE CHECK ===")
print(f"SVC(precomputed) versus original QSVC predictions: {agreement}/600")
if agreement != 600:
    raise RuntimeError("Mismatch with original QSVC C=1; STOP, no files written")

# Aggregate confusion matrices from observed predictions, with same fixed class order.
for C in C_GRID:
    subset = predictions_df.loc[predictions_df["C"] == C]
    cm = confusion_matrix(subset["actual"], subset["predicted"], labels=CLASSES)
    for i, actual in enumerate(CLASSES):
        for j, guess in enumerate(CLASSES):
            confusion_rows.append({
                "scope": "aggregate", "fold": "all", "C": float(C),
                "actual": actual, "predicted": guess, "count": int(cm[i, j]),
            })
confusion_df = pd.DataFrame(confusion_rows)
summary_df = (
    metrics_df.groupby("C", sort=True)
    .agg(
        folds=("fold", "count"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_sd=("macro_f1", "std"),
        accuracy_mean=("accuracy", "mean"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        recall_high_mean=("recall_high", "mean"),
        kernel_train_seconds_mean=("kernel_train_seconds", "mean"),
        kernel_val_seconds_mean=("kernel_val_seconds", "mean"),
        svc_fit_seconds_mean=("svc_fit_seconds", "mean"),
        svc_predict_seconds_mean=("svc_predict_seconds", "mean"),
    )
    .reset_index().sort_values("macro_f1_mean", ascending=False)
)
assert summary_df["folds"].eq(5).all()
assert metrics_df["macro_f1"].between(0,1).all()
assert metrics_df["accuracy"].between(0,1).all()
assert metrics_df["recall_high"].between(0,1).all()
assert np.isfinite(metrics_df.select_dtypes(include="number").to_numpy()).all()

# No automatic freeze of the best-performing C.
suggested_C = float(summary_df.iloc[0]["C"])
print("\n=== QSVC C-GRID — 5-FOLD RESULTS ===")
print(summary_df[["C", "macro_f1_mean", "macro_f1_sd", "accuracy_mean",
                  "balanced_accuracy_mean", "recall_high_mean"]].round(4).to_string(index=False))
print(f"[INFO] Highest observed CV Macro-F1: C={suggested_C:g} (candidate only; NOT frozen)")

config = {
    "experiment": "E01_QSVC_k4_C_grid_exploratory",
    "status": "EXPLORATORY_POST_BASELINE_NOT_PREREGISTERED",
    "reason": "C grid was added after initial C=1 QSVC CV results were observed",
    "C_grid": list(C_GRID),
    "suggested_C_by_CV_macro_f1_not_frozen": suggested_C,
    "original_primary_QSVC_C": 1.0,
    "class_weight": None,
    "training_pool": "quantum_dev_pool_600",
    "folds": 5,
    "train_n_per_fold": 480,
    "validation_n_per_fold": 120,
    "pca_k": 4,
    "features": approved,
    "preprocessing": "fit_preprocessor_PCA_MinMax_on_train_only_clip_0_pi",
    "feature_map": {"type": "zz_feature_map", "qubits": 4, "reps": 2, "entanglement": "linear"},
    "quantum_kernel": "FidelityStatevectorKernel",
    "shots": None,
    "kernel_cache": "in_memory_per_fold_reused_for_all_C; no disk cache",
    "classifier": "sklearn.svm.SVC(kernel='precomputed')",
    "C1_prediction_equivalence_count": agreement,
    "C1_prediction_equivalence_total": 600,
    "time_note": "Kernel train/val times shared among all C in a fold; do not sum 3 times",
    "metric_selection": "Macro-F1 exploratory CV; do not claim unbiased tuned-C CV performance",
    "final_test_used": False,
    "versions": {name: version(name) for name in
                 ["qiskit", "qiskit-machine-learning", "scikit-learn", "numpy", "pandas"]},
    "hashes": {
        "split_manifest": sha256_file(manifest_path),
        "feature_policy": sha256_file(ROOT / "configs/feature_policy.yaml"),
        "original_C1_predictions": sha256_file(BASELINE),
    },
}

# Refuse overwrites again immediately before writing.
existing = [str(p) for p in FILES.values() if p.exists()]
if existing:
    raise FileExistsError(f"Benchmark outputs already exist; no overwrite: {existing}")
OUT.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(FILES["metrics"], index=False)
predictions_df.to_csv(FILES["predictions"], index=False)
confusion_df.to_csv(FILES["confusion"], index=False)
summary_df.to_csv(FILES["summary"], index=False)
with FILES["config"].open("x", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2, allow_nan=False)
print("\n[PASS] C=1 identical to original QSVC over 600 validation samples")
print("[PASS] Saved 5 C-grid files to docs/results/")
print("[INFO] Final Test untouched; no model selection frozen")


=== C=1 EQUIVALENCE CHECK ===
SVC(precomputed) versus original QSVC predictions: 600/600

=== QSVC C-GRID — 5-FOLD RESULTS ===
   C  macro_f1_mean  macro_f1_sd  accuracy_mean  balanced_accuracy_mean  recall_high_mean
10.0         0.4224       0.0463         0.5083                  0.4194            0.2242
 1.0         0.3603       0.0167         0.5633                  0.3879            0.0000
 0.1         0.2347       0.0010         0.5433                  0.3333            0.0000
[INFO] Highest observed CV Macro-F1: C=10 (candidate only; NOT frozen)

[PASS] C=1 identical to original QSVC over 600 validation samples
[PASS] Saved 5 C-grid files to docs/results/
[INFO] Final Test untouched; no model selection frozen


## Kết luận và quy tắc dùng kết quả

- **C=1** là baseline QSVC đã có trước đó; notebook xác minh 600/600 dự đoán khớp với `QSVC` gốc mới coi kết quả C-grid tương thích.
- Các `C=0.1` và `C=10` là **sensitivity/tuning exploratory**. Nếu dùng để chọn C từ chính 5 folds, không báo điểm trung bình C được chọn như một ước lượng độc lập không thiên lệch sau tuning. Giữ nguyên báo cáo primary C=1 và báo cáo grid riêng, trừ khi nhóm chính thức điều chỉnh và tài liệu hóa protocol.
- `kernel_train_seconds` và `kernel_val_seconds` được **dùng chung giữa 3 C**, chỉ tính một lần mỗi fold; không cộng 3 lần để so quantum runtime với classical.
- **Không động đến Final Test**, không xóa script QSVC cũ cho tới khi notebook này đã PASS và những phần tái lập khác hoàn tất.
- Gửi output `C=1 EQUIVALENCE CHECK` + `QSVC C-GRID — 5-FOLD RESULTS` để kiểm tra trước khi chuyển checkpoint khác.
